# Project A Lab：从趋势信号到可交易组合

这是一个可运行的两小时实验，不是阅读笔记。你会先固定研究合同，再改变一个参数，观察 **signal → target weight → held position → PnL → cost** 每一层的变化。

**运行规则**

1. 先运行 baseline 单元格，不要先调参；
2. 在实验单元格的 `TODO` 处写下预期，再运行；
3. 只改变一个假设；
4. 把图和数字写入 `reports/memo_template.md`。

默认数据是确定性合成跨资产价格。它用来练习时间、单位、成本和研究证据，不能被表述成真实历史业绩。

In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import pandas as pd

# Make the notebook runnable from the project directory, research directory, or repository root.
PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "data.py").exists() and (p / "src" / "backtest.py").exists()
)
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / ".matplotlib"))

from src.data import DataConfig, make_prices
from src.features import make_features, signal_to_weights
from src.backtest import run_backtest
from src.metrics import summarize
from src.run import main as run_baseline

LAB_OUTPUT = PROJECT / "reports" / "lab_alpha_outputs"
LAB_OUTPUT.mkdir(parents=True, exist_ok=True)
metrics = run_baseline(output_dir=LAB_OUTPUT)
pd.DataFrame(metrics).T.round(4)

## 1. 先读懂基线的时间顺序

对任意日期 `t`，本项目的定义是：

```text
prices through t → signal/target weight at t → held weight at t+1 → return at t+1 → cost
```

`weights.csv` 是目标/持有计划，`backtest.py` 中的 `weights.shift(1)` 才是当天真正用于收益计算的仓位。先用下面的图确认：收益曲线使用的是净收益，成本上升是否只会压低结果。

In [ ]:
import matplotlib.pyplot as plt

prices = make_prices(DataConfig())
features = make_features(prices, lookback=60)
weights = signal_to_weights(features["momentum"], features["volatility"], rebalance_every=20)
curves = {}
for cost_bps in (0.0, 10.0, 25.0, 50.0):
    curves[f"trend {int(cost_bps)} bps"] = run_backtest(prices, weights, cost_bps=cost_bps)["equity"]
benchmark_weights = pd.DataFrame(1 / len(prices.columns), index=prices.index, columns=prices.columns)
curves["equal-weight 10 bps"] = run_backtest(prices, benchmark_weights, cost_bps=10)["equity"]

ax = pd.DataFrame(curves).plot(figsize=(11, 5), logy=True, title="Project A: net equity and transaction-cost pressure")
ax.set_ylabel("equity (log scale)")
ax.grid(alpha=0.25)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "lab_alpha_equity.png", dpi=150)
plt.show()

summary = pd.DataFrame({name: summarize(run_backtest(prices, weights if name.startswith("trend") else benchmark_weights, cost_bps=(10 if "equal" in name else float(name.split()[1]))) ["net_return"]) for name in curves}).T
summary.round(4)

## 2. 参数实验：只改一个研究假设

**TODO（运行前写在这里或 memo 中）**：如果 lookback 从 60 改成 20/120，预计 signal turnover、收益和成本后的 Sharpe 怎样变化？理由是什么？

下面的矩阵固定数据、资产、权重上限和成本，只改变 `lookback`。图中的每个点都是完整回测，不要只挑最好的一点。

In [ ]:
lookbacks = [20, 60, 120]
costs = [0, 10, 25, 50]
rows = []
for lookback in lookbacks:
    local_features = make_features(prices, lookback=lookback)
    local_weights = signal_to_weights(local_features["momentum"], local_features["volatility"], rebalance_every=20)
    for cost_bps in costs:
        result = run_backtest(prices, local_weights, cost_bps=cost_bps)
        record = summarize(result["net_return"])
        record.update(lookback=lookback, cost_bps=cost_bps, average_turnover=float(result["turnover"].mean()))
        rows.append(record)
sensitivity = pd.DataFrame(rows)
print(sensitivity.pivot(index="lookback", columns="cost_bps", values="sharpe").round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lookback, group in sensitivity.groupby("lookback"):
    axes[0].plot(group["cost_bps"], group["annual_return"], marker="o", label=f"lookback={lookback}")
    axes[1].plot(group["cost_bps"], group["average_turnover"], marker="o", label=f"lookback={lookback}")
axes[0].set(title="Net annual return vs cost", xlabel="cost (bps)", ylabel="annual return")
axes[1].set(title="Turnover is independent of bps", xlabel="cost (bps)", ylabel="average daily turnover")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "lab_alpha_cost_sensitivity.png", dpi=150)
plt.show()

## 3. 暴露审计：权重不是 signal

`momentum / volatility` 是 signal，归一化、cap 和 forward-fill 后才成为 target weight。请观察 cap 如何改变 gross exposure；单名权重满足上限，不代表组合已经风险中性。

In [ ]:
exposure = pd.DataFrame({
    "gross_exposure": weights.abs().sum(axis=1),
    "net_exposure": weights.sum(axis=1),
    "largest_abs_weight": weights.abs().max(axis=1),
    "active_assets": (weights.abs() > 0).sum(axis=1),
})
print(exposure.describe().round(4))
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
exposure[["gross_exposure", "net_exposure"]].plot(ax=axes[0], title="Exposure through time")
exposure["largest_abs_weight"].plot(ax=axes[1], title="Largest absolute name weight", color="tab:orange")
axes[1].axhline(0.25, color="black", linestyle="--", label="cap")
axes[1].legend()
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "lab_alpha_exposure.png", dpi=150)
plt.show()

## 4. 交付检查

在 `reports/memo_template.md` 中完成：

- [ ] 研究问题、决策时点、benchmark、成本区间；
- [ ] 一张 equity/cost 图和一张 exposure 图；
- [ ] lookback × cost 表，并说明预期是否被支持；
- [ ] 一个“结果不支持部署”的条件；
- [ ] `lab_alpha_outputs/run_manifest.json` 中的配置和产物 hash。

面试表达：先讲信息时点和 held weights，再讲 alpha 直觉，最后讲成本、容量和失败状态。